# Faruq-v3 — CIR0 vs CPE0 Hard-Confusion Reduction Audit

Secondary post-training diagnostic only. No training. Validation-only inference is used to export GT-aligned object events for CPE0 and CIR0. The 17 undirected hard-confusion families are loaded from the consensus JSON that was frozen before Circle-CPE results.

**This audit cannot overturn the frozen screening decision `STOP_CIRCLE_CPE`. Test is never extracted/opened.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan GPU untuk mempercepat validation inference.'

BASE_REQUIRED = ('bundles/faruq-development-v3-grouped.tar',)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=BASE_REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, BASE_REQUIRED[0])
EXPERIMENTS = PROJECT_ROOT / 'experiments'

def find_unique_checkpoint(run_name):
    matches = [p for p in EXPERIMENTS.rglob('best.pt') if p.parent.name == 'weights' and p.parent.parent.name == run_name]
    if len(matches) != 1:
        raise RuntimeError(f'{run_name}: expected exactly 1 best.pt, found {len(matches)}: {matches}')
    return matches[0]

def find_unique_named_file(filename, parent_hint=None):
    matches = list(EXPERIMENTS.rglob(filename))
    if parent_hint is not None:
        matches = [p for p in matches if parent_hint in p.parts]
    if len(matches) != 1:
        raise RuntimeError(f'{filename}: expected exactly 1 file, found {len(matches)}: {matches}')
    return matches[0]

CPE0_CHECKPOINT = find_unique_checkpoint('CPE0_seed42')
CIR0_CHECKPOINT = find_unique_checkpoint('CIR0_seed42')
CONSENSUS_JSON = find_unique_named_file('cross_model_hard_confusion_consensus.json', 'faruq-v3-cross-model-hard-confusion-consensus-seed42-v1')

DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'

OUTPUT_ROOT = EXPERIMENTS / 'faruq-v3-circle-cpe-hard-confusion-reduction-v1'
EVENT_DIR = OUTPUT_ROOT / 'events'
EVENT_DIR.mkdir(parents=True, exist_ok=True)
CPE0_EVENT = EVENT_DIR / 'CPE0_seed42_events.json'
CIR0_EVENT = EVENT_DIR / 'CIR0_seed42_events.json'
SUMMARY = OUTPUT_ROOT / 'cir0_vs_cpe0_hard_confusion_reduction.json'

print('GPU:', torch.cuda.get_device_name(0))
print('CPE0:', CPE0_CHECKPOINT)
print('CIR0:', CIR0_CHECKPOINT)
print('CONSENSUS:', CONSENSUS_JSON)
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_circle_cpe_hard_confusion_reduction.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
def export_if_missing(model_name, checkpoint, output):
    if output.is_file():
        print(model_name, 'event already exists -> reuse:', output)
        return
    command = [
        sys.executable, str(REPO/'scripts/export_validation_object_events.py'),
        '--checkpoint', str(checkpoint),
        '--data-root', str(DATA_ROOT),
        '--output', str(output),
        '--model-name', model_name,
        '--seed', '42', '--device', '0',
    ]
    print('EXPORT', model_name)
    subprocess.run(command, cwd=REPO, check=True)

export_if_missing('CPE0', CPE0_CHECKPOINT, CPE0_EVENT)
export_if_missing('CIR0', CIR0_CHECKPOINT, CIR0_EVENT)


In [ ]:
command = [
    sys.executable, '-m', 'coffee_detector.analysis.circle_cpe_hard_confusion_reduction',
    '--cpe0-event', str(CPE0_EVENT),
    '--cir0-event', str(CIR0_EVENT),
    '--consensus-json', str(CONSENSUS_JSON),
    '--output', str(SUMMARY),
]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
assert result['screening_decision_remains'] == 'STOP_CIRCLE_CPE'


In [ ]:
import pandas as pd
from IPython.display import display

print('CLASSIFICATION ERRORS @ IoU50:', result['classification_errors_iou50'])
print('FROZEN HARD-FAMILY ERRORS:', result['frozen_hard_family_errors'])
print('PAIRED TRANSITIONS:', result['paired_transitions'])
print('FAMILY SUMMARY:', result['family_summary'])
print('SCREENING DECISION REMAINS:', result['screening_decision_remains'])

df = pd.DataFrame(result['families'])
display(df[['family','cpe0_errors','cir0_errors','reduction','delta_cir0_minus_cpe0']].head(17))
print('SUMMARY:', SUMMARY)
print('Kirim seluruh blok ringkasan + tabel 17 family. Jangan membuka test.')
